# Dummy Performance Comparison

Compares Performance of `RazorsEdgeDummyTask` with `BaseBatchedDummyTask` under load.


In [1]:
from pathlib import Path
import sys
import psutil
import os

repo_root = Path.cwd()

while not (repo_root / "src").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not find 'src' directory in any parent")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

p = psutil.Process(os.getpid())
if psutil.WINDOWS:
    p.nice(psutil.HIGH_PRIORITY_CLASS)
else:
    try:
        p.nice(-10)
    except psutil.AccessDenied:
        print("Elevation (sudo) is required to set high priority on Unix.")

In [2]:
import asyncio
import time
import matplotlib.pyplot as plt
from src.executor.process_manager import ComputeExecutor
from demos.synthetic.razors_edge_dummy_task import RazorsEdgeDummyTask
from demos.synthetic.base_batched_dummy_task_variants import BaseBatchedDummyTask, BaseBatchedDummyTaskB2, BaseBatchedDummyTaskB3, BaseBatchedDummyTaskB4



## Start executor

In [3]:
executor = ComputeExecutor(
    [RazorsEdgeDummyTask, BaseBatchedDummyTask, BaseBatchedDummyTaskB2, BaseBatchedDummyTaskB3, BaseBatchedDummyTaskB4],
    async_limit=64,
    model_thread_limit=1,
)


## Basic functionality check

In [4]:
sample = executor.sync_compute_fn(RazorsEdgeDummyTask, "hello razors edge")
assert isinstance(sample, list)
assert len(sample) > 0
assert isinstance(sample[0], float)
print("sync check 1 passed")

sample = executor.sync_compute_fn(BaseBatchedDummyTask, "hello razors edge")
assert isinstance(sample, list)
assert len(sample) > 0
assert isinstance(sample[0], float)
print("sync check 2 passed")


sync check 1 passed
sync check 2 passed


## Benchmark helpers

In [5]:
import random
import string

def generate_random_strings(n, a, b, seed=42):
    random.seed(seed)  # set seed for reproducibility
    chars = string.ascii_letters + string.digits  # a-z, A-Z, 0-9
    result = []
    for _ in range(n):
        length = random.randint(a, b)
        rand_str = ''.join(random.choice(chars) for _ in range(length))
        result.append(rand_str)
    return result

async def benchmark_async(target, max_token_count: int, request_count: int) -> tuple[float, float]:
    payloads = generate_random_strings(request_count, 1, max_token_count)
    start = time.perf_counter()

    await asyncio.gather(
        *(executor.async_compute_fn(target, payload) for payload in payloads)
    )
    elapsed = time.perf_counter() - start
    
    return elapsed, request_count / elapsed


## Run timing benchmarks

In [6]:
max_token_size = 1000
n_req = 200
tasks = [RazorsEdgeDummyTask, BaseBatchedDummyTask, BaseBatchedDummyTaskB2, BaseBatchedDummyTaskB3, BaseBatchedDummyTaskB4]

rps_list = []
for task in tasks:
    a_elapsed, a_rps = await benchmark_async(task, max_token_size, n_req)
    rps_list.append(a_rps)
    print(f"Completeded one benchmark for {task.__name__}", a_rps)

Completeded one benchmark for RazorsEdgeDummyTask 13.273876983548634
Completeded one benchmark for BaseBatchedDummyTask 11.376095579861907


CancelledError: 

## Matplotlib diagram 1: Throughput vs Load (Parallelism Limit)

In [ ]:
plt.bar([i.__name__ for i in tasks], rps_list)
plt.ylabel("RPS")
plt.xticks(rotation=45, ha='right')
plt.xlabel("task")
plt.title("Synthetic Load Basic Benchmarks")
plt.savefig(Path("..") / ".." / "images" / f"{plt.gca().get_title()}.png", bbox_inches='tight')
plt.show()